# LivePortrait GStreamer Plugin Tutorial

This notebook demonstrates how to use the `gst-liveportrait` plugin for real-time head reenactment using our pre-built Docker environment. 

> **Note:** Output videos now include audio from the driving source.

## 1. Prerequisites

Ensure you have the TensorRT engines exported as described in the `README.md`. You will need Docker and NVIDIA Container Toolkit installed on your host machine.

## 2. Compile the Plugin

Even with the pre-built image, you must compile the C++ source code for your specific architecture inside the container.

In [ ]:
!docker run --rm -v $(pwd)/..:/workspace -w /workspace ducksouplab/liveportrait_gst:latest bash -c "mkdir -p build && cd build && cmake .. && make -j$(nproc)"

## 3. Basic Reenactment (with Audio)

The `liveportrait_process.py` wrapper automatically muxes audio from the driving video into the output.

In [ ]:
# Reenacting Leo
!python3 ../liveportrait_process.py \
    --input ../assets/video_example.mp4 \
    --output ../outputs/leo_std.mp4 \
    --source ../assets/leo.jpg \
    --config ../checkpoints/

In [ ]:
# Reenacting Petter
!python3 ../liveportrait_process.py \
    --input ../assets/video_example.mp4 \
    --output ../outputs/petter_std.mp4 \
    --source ../assets/Petter-Johansson.jpg \
    --config ../checkpoints/

## 4. Dynamic Eye Retargeting

This feature allows you to override driving eye motion with manual control (e.g., forced blinking).

### Step 4.1: Download and Compile Eye Engine
Ensure you have the official `eyeblink.engine` in your checkpoints.

In [ ]:
!wget -nc https://huggingface.co/warmshao/FasterLivePortrait/resolve/main/liveportrait_onnx/stitching_eye.onnx -O ../stitching_eye.onnx
!docker run --rm --gpus all -v $(pwd)/..:/workspace -w /workspace ducksouplab/liveportrait_gst:latest python3 compile_trt.py

### Step 4.2: Run with Forced Blink
We use `--enable-eye-retargeting` and set `--eyes-open-ratio 0.0` with `--eye-retargeting-strength 1.5`.

In [ ]:
!python3 ../liveportrait_process.py \
    --input ../assets/video_example.mp4 \
    --output ../outputs/leo_blink.mp4 \
    --source ../assets/leo.jpg \
    --config ../checkpoints/ \
    --enable-eye-retargeting \
    --eyes-open-ratio 0.0 \
    --eye-retargeting-strength 1.5

## 5. Manual GStreamer Audio Pipeline

If running manually, use this multi-branch pipeline to preserve audio:

In [ ]:
!docker run --rm --gpus all -v $(pwd)/..:/workspace -w /workspace ducksouplab/liveportrait_gst:latest bash -c "\
    GST_PLUGIN_PATH=./build gst-launch-1.0 \
    filesrc location=assets/video_example.mp4 name=src ! decodebin name=dec \
    dec. ! queue ! videoconvert ! videocrop left=280 right=280 ! videoscale ! video/x-raw,width=512,height=512,format=RGB ! \
    liveportrait config-path=./checkpoints source-image=assets/test_image.jpg ! \
    videoconvert ! x264enc bitrate=2000 ! mux. \
    dec. ! queue ! audioconvert ! audioresample ! avenc_aac ! mux. \
    mp4mux name=mux ! filesink location=outputs/manual_audio_output.mp4"